# QCM region analysis: view-run

This notebook was generated from QCM Viewer for one current range or saved region. It starts with the quantity selected in the UI, then adds QCM-D overview plots, statistics, a representative raw sweep, and overlapping saved regions.

In [1]:
# Environment and imports
# Run this cell first. It installs missing packages into THIS Jupyter kernel.
from pathlib import Path
import importlib
import subprocess
import sys

PROJECT_ROOT = Path(r"/Users/albincarlsson/Documents/kemi_jobb/QCM_Viewer2").resolve()
RUN_PATH = Path(r"view-run").resolve()
T0 = 1779806363650633
T1 = 1779806366890796
GROUPS = [0, 1, 2, 3, 4]
ORDERS = {0: 1, 1: 3, 2: 5, 3: 7, 4: 9}
COLUMNS = ['fit_center', 'fit_fwhm', 'fit_gamma']
REGION_LABEL = 'cool!'
SELECTED_QUANTITY = 'fit_fwhm'
US = 1_000_000

REQUIRED = [
    ("polars", "polars"),
    ("pandas", "pandas"),
    ("holoviews", "holoviews"),
    ("hvplot", "hvplot.polars"),
    ("bokeh", "bokeh"),
    ("pyarrow", "pyarrow"),
    ("duckdb", "duckdb"),
    ("jupyter-bokeh", "jupyter_bokeh"),
]

def ensure_import(package_name, import_name):
    try:
        return importlib.import_module(import_name)
    except Exception:
        print("Installing missing package into this notebook kernel:", package_name)
        subprocess.check_call([sys.executable, "-m", "pip", "install", package_name])
        return importlib.import_module(import_name)

for package_name, import_name in REQUIRED:
    ensure_import(package_name, import_name)

try:
    import qcm
except Exception:
    if PROJECT_ROOT.exists():
        print("Installing local qcm package into this notebook kernel:", PROJECT_ROOT)
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", str(PROJECT_ROOT)])
        import qcm
    else:
        raise RuntimeError(
            "Could not import qcm and the original project folder was not found. "
            "Open Jupyter from the qcm_refactor folder, or run: python -m pip install -e /path/to/qcm_refactor"
        )

import polars as pl
import pandas as pd
import holoviews as hv
import hvplot.polars  # registers .hvplot on Polars DataFrames
from qcm.viz import science
from qcm.viz.theme import QUANTITIES, quantity

hv.extension("bokeh")

run = qcm.open_run(RUN_PATH)
span_s = (T1 - T0) / US
print("Run:", run.id)
print("Region:", REGION_LABEL)
print("Selected UI quantity:", SELECTED_QUANTITY)
print(f"Window: {span_s:.3f} s")
print("Groups:", GROUPS)


%opts magic unavailable (pyparsing cannot be imported)
%compositor magic unavailable (pyparsing cannot be imported)


Run: view-run
Region: cool!
Selected UI quantity: fit_fwhm
Window: 3.240 s
Groups: [0, 1, 2, 3, 4]


## Load exported region

In [2]:
# Load raw fit data for this exact exported region.
raw = run.timeline(COLUMNS, t0=T0, t1=T1, groups=GROUPS, level="raw")
raw = raw.with_columns(((pl.col("timestamp") - T0) / US).alias("elapsed_s"))
print(f"Rows: {raw.height:,}")
raw.head()


Rows: 1,520


timestamp,group,fit_center,fit_fwhm,fit_gamma,elapsed_s
i64,i64,f64,f64,f64,f64
1779806363651382,0,4958712.5,1583.741455,791.870728,0.000749
1779806363651382,1,1.4853603e7,1505.810181,752.90509,0.000749
1779806363651382,2,2.4752082e7,2468.109863,1234.054932,0.000749
1779806363651382,3,3.464962e7,3510.909668,1755.454834,0.000749
1779806363651382,4,4.454826e7,4464.950684,2232.475342,0.000749


## Zero/reference range used inside this notebook

In [3]:
# Helper functions used in this notebook.
def add_elapsed(df: pl.DataFrame) -> pl.DataFrame:
    if df.is_empty() or "timestamp" not in df.columns:
        return df
    return df.with_columns(((pl.col("timestamp") - T0) / US).alias("elapsed_s"))

# Default notebook zero/reference range: first 10% of the exported region.
# Change BASELINE_START_S / BASELINE_END_S and re-run cells below if needed.
BASELINE_START_S = 0.0
BASELINE_END_S = max(span_s * 0.10, min(span_s, 1.0))
B0 = int(T0 + BASELINE_START_S * US)
B1 = int(T0 + BASELINE_END_S * US)
print(f"Notebook zero/reference range: {BASELINE_START_S:.3f}–{BASELINE_END_S:.3f} s")


Notebook zero/reference range: 0.000–1.000 s


## Selected UI quantity

In [4]:
# Plot the same quantity that was selected in the UI when this notebook was exported.
q = quantity(SELECTED_QUANTITY)
main_selected = run.timeline(list(q.sources), t0=T0, t1=T1, groups=GROUPS)
baseline_selected = None
if q.referenced:
    baseline_selected = run.timeline(list(q.sources), t0=B0, t1=B1, groups=GROUPS, level="raw")
selected_df = add_elapsed(science.compute(main_selected, SELECTED_QUANTITY, ORDERS, baseline_df=baseline_selected))
selected_title = f"{q.label} — {REGION_LABEL}"
selected_plot = selected_df.hvplot.line(
    x="elapsed_s", y="value", by="group", responsive=True, height=420,
    title=selected_title, xlabel="Time in exported region [s]", ylabel=q.axis_label,
).opts(active_tools=["wheel_zoom"], tools=["hover", "crosshair", "box_zoom", "reset"])
selected_plot


:NdOverlay   [group]
   :Curve   [elapsed_s]   (value)

## Compute QCM-D quantities

In [5]:
# Compute canonical QCM-D quantities.
baseline = run.timeline(["fit_center", "fit_fwhm"], t0=B0, t1=B1, groups=GROUPS, level="raw")
main = run.timeline(["fit_center", "fit_fwhm"], t0=T0, t1=T1, groups=GROUPS)

df_norm = add_elapsed(science.compute(main, "delta_f_norm", ORDERS, baseline_df=baseline))
dD = add_elapsed(science.compute(main, "delta_D", ORDERS, baseline_df=baseline))

qcmd_long = pl.concat([
    df_norm.with_columns(pl.lit("delta_f_norm").alias("quantity_key"), pl.lit("Δf/n [Hz]").alias("quantity")),
    dD.with_columns(pl.lit("delta_D").alias("quantity_key"), pl.lit("ΔD [×10⁻⁶]").alias("quantity")),
], how="diagonal")
qcmd_long.head()


timestamp,group,value,elapsed_s,quantity_key,quantity
i64,i64,f64,f64,str,str
1779806363651382,0,179.648936,0.000749,"""delta_f_norm""","""Δf/n [Hz]"""
1779806363651382,1,169.255319,0.000749,"""delta_f_norm""","""Δf/n [Hz]"""
1779806363651382,2,153.493617,0.000749,"""delta_f_norm""","""Δf/n [Hz]"""
1779806363651382,3,136.224924,0.000749,"""delta_f_norm""","""Δf/n [Hz]"""
1779806363651382,4,141.815603,0.000749,"""delta_f_norm""","""Δf/n [Hz]"""


## Statistics for all quantities

In [6]:
# Rich per-group statistics for every registered quantity in this exported region.
stat_frames = []
for key in QUANTITIES:
    q = quantity(key)
    source_cols = list(q.sources)
    main_q = run.timeline(source_cols, t0=T0, t1=T1, groups=GROUPS)
    baseline_q = None
    if q.referenced:
        baseline_q = run.timeline(source_cols, t0=B0, t1=B1, groups=GROUPS, level="raw")
    values = science.compute(main_q, key, ORDERS, baseline_df=baseline_q)
    s = science.summary_stats(values).with_columns(
        pl.lit(key).alias("quantity_key"),
        pl.lit(q.label).alias("quantity"),
        pl.lit(q.unit or "—").alias("unit"),
    )
    stat_frames.append(s)

all_stats = pl.concat(stat_frames, how="diagonal") if stat_frames else pl.DataFrame()
all_stats


group,rows,valid,missing,start_us,end_us,duration_s,first,last,net_change,abs_net_change,mean,median,std,variance,min,q01,q05,q10,q25,q75,q90,q95,q99,max,mean_abs,rms,mean_abs_step,step_std,sum_abs,sum,range,iqr,sem,slope_per_s,cv,time_average_proxy,quantity_key,quantity,unit
i64,u32,u32,u32,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str,str
0,304,304,0,1779806363651382,1779806366883168,3.231786,179.648936,178.648936,-1.0,1.0,-175.53034,-179.601064,350.708058,122996.141846,-869.351064,-862.351064,-830.351064,-742.851064,-452.351064,178.148936,183.148936,186.648936,190.648936,191.648936,313.546508,391.666248,9.40264,13.657045,95318.138298,-53361.223404,1061.0,630.5,20.114487,-0.309426,1.997991,-16511.372784,"""delta_f_norm""","""Δf / n""","""Hz"""
1,304,304,0,1779806363651382,1779806366883168,3.231786,169.255319,167.255319,-2.0,2.0,-85.96069,-125.244681,227.934024,51953.919083,-458.078014,-454.411348,-441.411348,-417.744681,-283.744681,164.588652,166.588652,167.255319,168.255319,169.255319,215.134309,243.253485,6.033003,9.04286,65400.829787,-26132.049645,627.333333,448.333333,13.072913,-0.618853,2.651608,-8085.946794,"""delta_f_norm""","""Δf / n""","""Hz"""
2,304,304,0,1779806363651382,1779806366883168,3.231786,153.493617,153.093617,-0.4,0.4,-47.572172,-82.706383,184.030805,33867.337176,-321.306383,-316.106383,-310.106383,-295.306383,-217.306383,151.893617,167.893617,177.893617,182.293617,182.293617,174.955235,189.786836,5.20264,7.839009,53186.391489,-14461.940426,503.6,369.2,10.554891,-0.123771,3.868455,-4474.906577,"""delta_f_norm""","""Δf / n""","""Hz"""
3,304,304,0,1779806363651382,1779806366883168,3.231786,136.224924,158.510638,22.285714,22.285714,-25.604023,-60.917933,159.252266,25361.28431,-262.06079,-255.203647,-238.06079,-229.489362,-173.489362,136.796353,173.367781,175.653495,177.93921,177.93921,150.928531,161.038582,4.505422,6.830442,45882.273556,-7783.6231,440.0,310.285714,9.133744,6.895789,6.219814,-2408.458698,"""delta_f_norm""","""Δf / n""","""Hz"""
4,304,304,0,1779806363651382,1779806366883168,3.231786,141.815603,157.815603,16.0,16.0,-22.900771,-64.628842,149.302768,22291.316431,-234.628842,-230.184397,-216.851064,-208.851064,-159.51773,139.593381,162.260047,165.815603,168.48227,168.48227,141.943636,150.805952,6.210488,9.502298,43150.865248,-6961.834515,403.111111,299.111111,8.563101,4.950823,6.519552,-2154.175591,"""delta_f_norm""","""Δf / n""","""Hz"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
0,304,304,0,1779806363651382,1779806366883168,3.231786,791.870728,864.506775,72.636047,72.636047,902.36146,854.740631,113.192269,12812.489656,791.870728,795.028381,800.334167,807.793335,825.488647,946.719299,1121.760742,1172.051758,1196.297119,1201.226685,902.36146,909.41,5.727424,8.241115,274317.883789,274317.883789,409.355957,121.230652,6.492022,22.475513,0.12544,84881.203084,"""fit_gamma""","""HWHM (Γ)""","""Hz"""
1,304,304,0,1779806363651382,1779806366883168,3.231786,752.90509,762.327698,9.422607,9.422607,1163.245441,1057.895203,417.120848,173989.801535,738.536377,740.523682,745.010742,749.479431,787.838928,1478.052124,1873.294434,1985.62793,2026.281006,2029.455566,1163.245441,1235.539325,9.767115,14.988997,353626.614075,353626.614075,1290.919189,690.213196,23.923521,2.915604,0.358584,109421.420253,"""fit_gamma""","""HWHM (Γ)""","""Hz"""
2,304,304,0,1779806363651382,1779806366883168,3.231786,1234.054932,1316.223999,82.169067,82.169067,1876.167733,1750.955566,618.788832,382899.619052,1226.942627,1231.776489,1234.966064,1239.484253,1351.67627,2397.939941,2935.161133,3058.87085,3114.541016,3127.950684,1876.167733,1975.258323,16.497863,25.018625,570354.990845,570354.990845,1901.008057,1046.263672,35.489974,25.425281,0.329815,176482.907855,"""fit_gamma""","""HWHM (Γ)""","""Hz"""


## QCM-D overview plots

In [7]:
# Interactive QCM-D overview for the exported region.
df_plot = df_norm.hvplot.line(
    x="elapsed_s", y="value", by="group", responsive=True, height=360,
    title=f"Δf/n — {REGION_LABEL}", xlabel="Time in exported region [s]", ylabel="Δf/n [Hz]",
).opts(active_tools=["wheel_zoom"], tools=["hover", "crosshair", "box_zoom", "reset"])
dD_plot = dD.hvplot.line(
    x="elapsed_s", y="value", by="group", responsive=True, height=360,
    title=f"ΔD — {REGION_LABEL}", xlabel="Time in exported region [s]", ylabel="ΔD [×10⁻⁶]",
).opts(active_tools=["wheel_zoom"], tools=["hover", "crosshair", "box_zoom", "reset"])
df_plot + dD_plot


:Layout
   .NdOverlay.I  :NdOverlay   [group]
      :Curve   [elapsed_s]   (value)
   .NdOverlay.II :NdOverlay   [group]
      :Curve   [elapsed_s]   (value)

In [8]:
# Df fingerprint: ΔD vs Δf/n.
fingerprint_df = df_norm.rename({"value": "delta_f_norm"}).join(
    dD.rename({"value": "delta_D"}), on=["timestamp", "group", "elapsed_s"], how="inner"
)
fingerprint_df.hvplot.line(
    x="delta_f_norm", y="delta_D", by="group", responsive=True, height=420,
    title=f"Df fingerprint — {REGION_LABEL}", xlabel="Δf/n [Hz]", ylabel="ΔD [×10⁻⁶]"
).opts(active_tools=["wheel_zoom"], tools=["hover", "crosshair", "box_zoom", "reset"])


:NdOverlay   [group]
   :Curve   [delta_f_norm]   (delta_D)

## Raw sweep check

In [9]:
# Inspect a raw sweep from the middle of the exported region.
idx = run.sweep_index(t0=T0, t1=T1, groups=GROUPS)
mid_sequence = int(idx["sequence"][idx.height // 2]) if idx.height else None
sweep = run.sweeps_at(sequence=mid_sequence, groups=GROUPS) if mid_sequence is not None else pl.DataFrame()
if not sweep.is_empty():
    display(sweep.hvplot.line(
        x="frequency", y=["conductance", "susceptance"], by="group", responsive=True, height=420,
        title=f"Raw resonance sweep {mid_sequence}", xlabel="Frequency [Hz]", ylabel="Signal [a.u.]"
    ).opts(active_tools=["wheel_zoom"], tools=["hover", "crosshair", "box_zoom", "reset"]))
    display(sweep.hvplot.scatter(
        x="raw_i", y="raw_q", by="group", responsive=True, height=420, size=5,
        title=f"I/Q sweep {mid_sequence}", xlabel="Raw I [a.u.]", ylabel="Raw Q [a.u.]"
    ).opts(active_tools=["wheel_zoom"], tools=["hover", "box_zoom", "reset"]))
else:
    print("No sweep found for this region.")


:NdOverlay   [Variable]
   :Curve   [frequency]   (value)

:NdOverlay   [group]
   :Scatter   [raw_i]   (raw_q)

## Saved regions overlapping this region

In [10]:
# Saved regions that overlap this exported region.
saved_regions = run.annotations(t0=T0, t1=T1)
region_rows = []
for a in saved_regions:
    start_s = (a.t0 - T0) / US
    end_s = ((a.t1 or a.t0) - T0) / US
    region_rows.append({
        "label": a.label,
        "type": a.type,
        "kind": a.tags[0] if a.tags else "region",
        "start_s": start_s,
        "end_s": end_s,
        "duration_s": max(0, end_s - start_s),
        "groups": a.groups,
        "created_at": a.created_at,
    })
region_table = pl.DataFrame(region_rows) if region_rows else pl.DataFrame()
region_table


label,type,kind,start_s,end_s,duration_s,groups,created_at
str,str,str,f64,f64,f64,list[i64],str
"""cool!""","""range""","""note""",0.0,3.240163,3.240163,"[0, 1, … 4]","""2026-05-28T08:53:38.420097+00:…"


## Optional exports

In [11]:
# Optional exports from the notebook.
# raw.write_parquet("region_raw.parquet")
# selected_df.write_csv("selected_quantity_timeseries.csv")
# all_stats.write_csv("region_stats.csv")
# qcmd_long.write_csv("region_qcmd_timeseries.csv")
# region_table.write_csv("saved_regions.csv")
